# Trip Planner — with **Graxella**

Same `trip_planner` package as notebook 01. Same tools, same LangGraph app, same LLM. **The only two additions** are:

1. A **Rulebook** with one promoted substitution:  
   `search_hotels_v1 → find_accommodations` with recipe `field_map={"location": "city"}`.
2. A one-line `graxella.wrap(app, tools=…, store=…, rulebook=…)` around the compiled app.

Everything else is bit-for-bit the same. That is the point of the sklearn-style layout: the intelligence layer plugs in without touching agent code.

**Expected impact** (compare with notebook 01):
- `hotels_ok = True` for every trip. No user-visible drift.
- Zero LLM retry — the runtime is deterministic once the rule is promoted.
- Every run becomes an `ExperienceEpisode` in a SQLite ledger.
- A PROV-O JSON-LD audit bundle closes the loop.

**Prereqs.** `pip install -e .` (already done in this repo), `ollama serve` with `qwen2.5:3b`.

## 1. Setup — same imports as notebook 01, plus `graxella`

In [ ]:
import sys
from pathlib import Path

_examples_root = Path.cwd().parent
if str(_examples_root) not in sys.path:
    sys.path.insert(0, str(_examples_root))

from trip_planner.tools import TOOLS, TOOLS_BY_NAME
from trip_planner.agents import (build_app, SAMPLE_QUERIES,
                                 run_batch, summarize, check_ollama, MODEL)

import graxella
from graxella import (Proposal, Rulebook, SqliteExperienceStore,
                      HiddenAgendaRunner, audit_export)

print(f'graxella {graxella.__version__} from {Path(graxella.__file__).parent}')
print(f'Model: {MODEL}')
print(f'Tools: {[t.name for t in TOOLS]}')

In [ ]:
assert check_ollama(), 'Ollama is not reachable.'

## 2. Prepare a fresh Rulebook + Experience store

Deleting any prior state so the notebook is reproducible run-to-run.

In [ ]:
here = Path.cwd()
store_path    = here / '_artifacts_trip_store.db'
rulebook_path = here / '_artifacts_trip_rulebook.json'
audit_path    = here / '_artifacts_trip_audit.jsonld'
for p in (store_path, rulebook_path, audit_path):
    if p.exists():
        p.unlink()

store    = SqliteExperienceStore(store_path)
rulebook = Rulebook(path=rulebook_path)
print('Store    :', store_path.name)
print('Rulebook :', rulebook_path.name, f'({len(list(rulebook.all_rules()))} rules)')

## 3. Promote the drift-heal rule

In a real deployment this `Proposal` would come from `DocsMiner` (docs describing the migration) or `RuleDistiller` (mined from lived `(err, ok)` episode pairs). Here we hand-craft it in one cell to keep the demo self-contained. The rule is exactly what a human reviewer would approve after Graxella surfaced the migration.

In [ ]:
proposal = Proposal(
    id='prop_trip_hotels_1',
    kind='rule',
    subject='travel:search_hotels_v1->find_accommodations',
    change={
        'if_intent': 'trip_planning',
        'replace_skill': 'search_hotels_v1',
        'with_skill': 'find_accommodations',
        'recipe': {'field_map': {'location': 'city'}},
    },
    evidence='hotels_v1 API sunset Q2 2026 — successor documented as find_accommodations(city, checkin, checkout)',
    derived_from=['docs:hotels_migration.md'],
    confidence=1.0,
)
rule = rulebook.promote(proposal, approved_by='ram@graxella')
print(f'Promoted {proposal.id} -> {rule.id}')
for r in rulebook.all_rules():
    print(f'  {r.replace_skill:22} -> {r.with_skill:22}  recipe={r.recipe}')

## 4. Wrap the same LangGraph app — one line

`graxella.wrap(...)` returns a `GraxellaApp` with the same `.invoke()` signature. Inside it, every tool's `invoke` is patched for the duration of the call so the rulebook decides the destination *before* the primary runs. `search_hotels_v1` is never actually executed — the wrapper dispatches directly to `find_accommodations` with the renamed arg.

In [ ]:
app = build_app()
smart_app = graxella.wrap(
    app,
    tools=TOOLS,
    store=store,
    rulebook=rulebook,
    intent='trip_planning',
    session_prefix='trip',
)
print('Wrapped app :', type(smart_app).__name__)
print('Underlying  :', type(smart_app.app).__name__)
print('Tools patched:', list(smart_app.tools_by_name))

## 5. Run the same three trips through the wrapped app

In [ ]:
first = SAMPLE_QUERIES[0]
print(f'REQUEST: {first}\n')
state = smart_app.invoke({'request': first})

print('PLAN       :', state['plan'])
print('FLIGHTS    :', state['flights'].split(chr(10))[0])
print('HOTELS     :', state['hotels'].split(chr(10))[0])
print('ACTIVITIES :', state['activities'].split(chr(10))[0])
print()
print('ERRORS     :', state.get('_errors'))
print('TIMINGS    :', state.get('_timings'))

In [ ]:
from IPython.display import Markdown, display
display(Markdown(state['itinerary']))

## 6. Batch metrics — side by side

Compare against notebook 01's `rows`. `hotels_ok` flips from `False` to `True`; `tool_errors` drops to 0.

In [ ]:
results = run_batch(smart_app)
rows = summarize(results)

try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    for r in rows:
        print(r)

n_total = len(rows)
n_hotels_ok = sum(1 for r in rows if r['hotels_ok'])
n_tool_errors = sum(r['tool_errors'] for r in rows)
total_wall = round(sum(r['wall_s'] for r in rows), 2)
print()
print(f'Trips run             : {n_total}')
print(f'Trips with hotels ok  : {n_hotels_ok} / {n_total}')
print(f'Total tool errors     : {n_tool_errors}')
print(f'Total wall time (s)   : {total_wall}')

## 7. Full itinerary — hotels populated

The concierge no longer has to apologize for missing accommodations.

In [ ]:
best_request, best_state, _ = results[0]
print(f'--- {best_request} ---')
display(Markdown(best_state['itinerary']))

## 8. The Experience ledger — every call recorded

One `ExperienceEpisode` per `smart_app.invoke(...)`. `find_accommodations=ok` on every hotels call — proving the substitution ran and the legacy tool was never touched.

In [ ]:
episodes = store.all()
print(f'Episodes: {len(episodes)}\n')
for e in episodes:
    tcs = ' | '.join(
        f"{tc.tool_id}={'ok' if tc.ok else 'ERR'}" for tc in e.tool_calls
    ) or '(no tools)'
    print(f'  {e.id[:14]}  session={e.session_id:8}  intent={e.intent:14}  tools=[{tcs}]')

## 9. Mine — should be empty because the rule is already promoted

In [ ]:
proposals = HiddenAgendaRunner(store=store).run()
print(f'Fresh proposals: {len(proposals)}')
for p in proposals:
    print(f'  [{p.kind}] {p.subject}  conf={p.confidence:.2f}')
if not proposals:
    print('(none — the substitute already lives in the rulebook, so the miner has nothing to say)')

## 10. Export the PROV-O audit bundle

The artifact an auditor reads. Nodes are joined by W3C PROV-O relations (`wasDerivedFrom`, `wasGeneratedBy`, `wasAssociatedWith`, `wasAttributedTo`).

Note: `audit_export` includes episodes **only when a proposal or rule cites them via `derived_from`**. Our proposal cites `docs:hotels_migration.md` (a docs-derived rule), so the breakdown below will show 0 episodes — the lived runs are still fully visible in the Experience ledger cell above. In a Phase-3 flow where `RuleDistiller` mines a proposal from `(err, ok)` episode pairs, `derived_from` would point at the episode ids and those episodes would appear in the bundle.

In [ ]:
import json
payload = audit_export(store, rulebook, [proposal], path=audit_path)
n_nodes = len(payload['@graph'])
kinds: dict[str, int] = {}
for node in payload['@graph']:
    kinds[node.get('grx:kind', '?')] = kinds.get(node.get('grx:kind', '?'), 0) + 1
print(f'Wrote {audit_path.name}  |  {audit_path.stat().st_size} bytes  |  {n_nodes} PROV nodes')
print(f'Breakdown: {json.dumps(kinds)}')

In [ ]:
# Peek at the first two audit graph nodes so a reader sees the shape.
for node in payload['@graph'][:2]:
    print(json.dumps(node, indent=2, default=str))
    print('---')

In [ ]:
store.close()

---

## Wrap-up — the delta at a glance

| Dimension | Notebook 01 (pure) | Notebook 02 (graxella) |
|---|---|---|
| `hotels_ok / total` | 0 / 3 | **3 / 3** |
| Tool errors | 3 | **0** |
| LLM retries | 0 (fails silently) | **0 (nothing to retry)** |
| Agent code changed | — | **0 lines** |
| Audit trail | none | **PROV-O JSON-LD bundle** |
| Extra state | none | **SQLite Experience ledger** |

**What actually changed** in going from notebook 01 to notebook 02:

- One `Rulebook` file on disk with a single promoted substitution.
- One line: `smart_app = graxella.wrap(app, tools=…, store=…, rulebook=…)`.

The agent code, the tool definitions, the LangGraph nodes, and the LLM prompts are **unchanged**. That is the entire value proposition: the intelligence layer plugs in around your graph without touching it, and everything it does is inspectable in the ledger and auditable via the PROV-O bundle.